# Ex. No. 4 – AWS SageMaker Console Setup

Build, train, deploy, evaluate, and clean up an XGBoost machine learning model using Amazon SageMaker.

## Step 1 – Create an Amazon SageMaker Notebook Instance

Create the notebook instance used to download and process data. Create an IAM role that allows SageMaker to access Amazon S3.

**Notebook instance name:** `SageMaker-Tutorial`  
**Notebook instance type:** `ml.t2.medium`  
**Elastic inference:** `none`

In Permissions and encryption, choose **Create a new role**, select **Any S3 bucket**, and create the role. Keep the remaining default settings and create the notebook instance. Wait until its status becomes **InService**, then choose **Open Jupyter** and create a `conda_python3` notebook.

## Step 2 – Prepare the Data

Run the following sections in new cells in order: create bucket → download data → split data → train model → deploy → evaluate → clean up.

In [ ]:
# Import Libraries and IAM Role Setup
import boto3, re, sys, math, json, os, sagemaker, urllib.request
from sagemaker import get_execution_role
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image
from IPython.display import display
from time import gmtime, strftime

# Define IAM role
role = get_execution_role()
prefix = 'sagemaker/DEMO-xgboost-dm'
my_region = boto3.session.Session().region_name  # set the region of the instance

# Automatically look up the XGBoost image URI and build an XGBoost container
xgboost_container = sagemaker.image_uris.retrieve("xgboost", my_region, "latest")

print("Success - the MySageMakerInstance is in the " + my_region + " region. You will use the "
      + xgboost_container + " container for your SageMaker endpoint.")

In [ ]:
# Create the S3 Bucket (unique name, auto-generated)
account_id = boto3.client('sts').get_caller_identity().get('Account')
bucket_name = 'sagemaker-demo-xgboost-{}-{}'.format(
    account_id, strftime('%Y%m%d%H%M%S', gmtime())
)

s3 = boto3.resource('s3')
try:
    if my_region == 'us-east-1':
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': my_region}
        )
    print('S3 bucket created successfully: ' + bucket_name)
except Exception as e:
    print('S3 error: ', e)

In [ ]:
# Download and Load the Dataset
try:
    urllib.request.urlretrieve(
        "https://d1.awsstatic.com/tmt/build-train-deploy-machine-learning-model-sagemaker/"
        "bank_clean.27f01fbbdf43271788427f3682996ae29ceca05d.csv",
        "bank_clean.csv"
    )
    print('Success: downloaded bank_clean.csv.')
except Exception as e:
    print('Data load error: ', e)

try:
    model_data = pd.read_csv('./bank_clean.csv', index_col=0)
    print('Success: Data loaded into dataframe.')
except Exception as e:
    print('Data load error: ', e)

In [ ]:
# Shuffle and Split the Data
# Training data = 70%, Test data = 30%
train_data, test_data = np.split(
    model_data.sample(frac=1, random_state=1729),
    [int(0.7 * len(model_data))]
)
print(train_data.shape, test_data.shape)

In [ ]:
# Train the XGBoost Model
# Reformat the header and first column of the training data, then upload it to S3.
# This is required for the Amazon SageMaker built-in XGBoost algorithm.
pd.concat([train_data['y_yes'], train_data.drop(['y_no', 'y_yes'], axis=1)],
          axis=1).to_csv('train.csv', index=False, header=False)

boto3.Session().resource('s3').Bucket(bucket_name).Object(
    os.path.join(prefix, 'train/train.csv')
).upload_file('train.csv')

s3_input_train = sagemaker.inputs.TrainingInput(
    s3_data='s3://{}/{}/train'.format(bucket_name, prefix),
    content_type='csv'
)

# Set up the SageMaker session, create the XGBoost estimator, and set hyperparameters
sess = sagemaker.Session()
xgb = sagemaker.estimator.Estimator(
    xgboost_container,
    role,
    instance_count=1,
    instance_type='ml.m4.xlarge',
    output_path='s3://{}/{}/output'.format(bucket_name, prefix),
    sagemaker_session=sess
)

In [ ]:
# Set XGBoost hyperparameters
xgb.set_hyperparameters(
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    verbosity=0,
    objective='binary:logistic',
    num_round=100
)

# Start the training job
xgb.fit({'train': s3_input_train})

In [ ]:
# Deploy the Model and Run Predictions
xgb_predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type='ml.m4.xlarge'
)

# Predict whether customers in the test data enrolled for the bank product or not
from sagemaker.serializers import CSVSerializer

test_data_array = test_data.drop(['y_no', 'y_yes'], axis=1).values
xgb_predictor.serializer = CSVSerializer()
predictions = xgb_predictor.predict(test_data_array).decode('utf-8')

# Safely parse the returned CSV string of predictions into a NumPy array
predictions_array = np.array(predictions.strip().split(','), dtype='float32')
print(predictions_array.shape)

In [ ]:
# Evaluate Model Performance
# Compare actual vs. predicted values in a confusion matrix
cm = pd.crosstab(
    index=test_data['y_yes'],
    columns=np.round(predictions_array),
    rownames=['Observed'],
    colnames=['Predicted']
)

tn = cm.iloc[0, 0]
fn = cm.iloc[1, 0]
tp = cm.iloc[1, 1]
fp = cm.iloc[0, 1]

p = (tp + tn) / (tp + tn + fp + fn) * 100

print("\n{0:<20}{1:<4.1f}%\n".format("Overall Classification Rate: ", p))
print("{0:<15}{1:<15}{2:>8}".format("Predicted", "No Purchase", "Purchase"))
print("Observed")
print("{0:<15}{1:<2.0f}% ({2:<}){3:>6.0f}% ({4:<})".format(
    "No Purchase", tn / (tn + fn) * 100, tn, fp / (tp + fp) * 100, fp
))
print("{0:<16}{1:<1.0f}% ({2:<}){3:>7.0f}% ({4:<}) \n".format(
    "Purchase", fn / (tn + fn) * 100, fn, tp / (tp + fp) * 100, tp
))

In [ ]:
# Clean Up Resources
# Terminate the resources used in this lab to avoid ongoing charges
xgb_predictor.delete_endpoint(delete_endpoint_config=True)

bucket_to_delete = boto3.resource('s3').Bucket(bucket_name)
bucket_to_delete.objects.all().delete()

# Then, from the SageMaker console:
# 1. Open Notebook instances
# 2. Select the notebook instance created for this tutorial -> Actions -> Stop
# 3. Once Status changes to 'Stopped' -> Actions -> Delete -> Delete